# Contexta Research — 01 Dataset Audit EDA

Built on SenseForge shared modules. Run sequentially from the repository root or notebooks directory. Missing data is explicitly skipped; no fabricated results.

In [2]:

from pathlib import Path
import os
import sys
import json

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

sys.path.insert(0, str(ROOT / "src"))

DATA_ROOT = Path("/Users/suvratsmacbookair/MyLocalFiles/NLP project")
os.environ["CONTEXTA_DATA_ROOT"] = str(DATA_ROOT)
os.environ["SENSEFORGE_DATA_ROOT"] = str(DATA_ROOT)

ART = ROOT / "artifacts"
ART.mkdir(parents=True, exist_ok=True)

print("Project:", ROOT)
print("Data:", DATA_ROOT)


Project: /Users/suvratsmacbookair/MyLocalFiles/NLP project/SenseForge
Data: /Users/suvratsmacbookair/MyLocalFiles/NLP project


## Audit actual dataset availability

In [4]:
from senseforge.data import audit,source_paths
from senseforge.research import comi_agreement,reservoir_parquet,infer_direction,extract_pair,digest,protected_hashes
report=audit();print(json.dumps(report,indent=2))

{
  "lid_train": {
    "available": true,
    "rows": 30487,
    "columns": [
      "Sentences",
      "Predicted tags",
      "Annotated by: Annotator 1",
      "Annotated by: Annotator 2",
      "Annotated by: Annotator 3"
    ],
    "path": "/Users/suvratsmacbookair/MyLocalFiles/NLP project/Comi-Lingua/LID_train.csv"
  },
  "lid_test": {
    "available": true,
    "rows": 5000,
    "columns": [
      "Sentences",
      "Predicted tags",
      "Annotated by: Annotator 1",
      "Annotated by: Annotator 2",
      "Annotated by: Annotator 3"
    ],
    "path": "/Users/suvratsmacbookair/MyLocalFiles/NLP project/Comi-Lingua/LID_test.csv"
  },
  "mt_train": {
    "available": true,
    "rows": 24692,
    "columns": [
      "Sentences",
      "Predicted_en_translation",
      "Predicted_RH_translation",
      "Predicted_DH_translation",
      "Annotator_1_en_translation",
      "Annotator_1_RH_translation",
      "Annotator_1_DH_translation",
      "Annotator_2_en_translation",
      "anno

## Human COMI annotation agreement

In [6]:
path=source_paths()['lid_train']
if path.exists(): print(json.dumps(comi_agreement(path),indent=2))
else: print('SKIPPED: COMI-LINGUA train CSV missing')

{
  "counts": {
    "sentences": 30487,
    "tokens": 588488,
    "unanimous": 533076,
    "majority": 586602,
    "unresolved": 1886,
    "alignment_errors": 413,
    "invalid_rows": 2
  },
  "majority_tag_counts": {
    "hi": 277494,
    "en": 93366,
    "ot": 36651,
    "e": 39970,
    "u": 13913,
    "h": 125208
  }
}


## Representative parquet samples and direction

In [8]:
samples={}
for name,path in source_paths().items():
    if path.exists() and path.suffix=='.parquet':
        result=reservoir_parquet(path,size=1000,seed=42)
        direction=infer_direction(result['sample']) if name.startswith('samanantar') else 'en-hi'
        samples[name]={'total_rows':result['total_rows'],'sampled':len(result['sample']),'direction':direction}
print(json.dumps(samples,indent=2))

{
  "iit_train": {
    "total_rows": 1659083,
    "sampled": 1000,
    "direction": "en-hi"
  },
  "iit_validation": {
    "total_rows": 520,
    "sampled": 520,
    "direction": "en-hi"
  },
  "iit_test": {
    "total_rows": 2507,
    "sampled": 1000,
    "direction": "en-hi"
  },
  "samanantar_0": {
    "total_rows": 1265714,
    "sampled": 1000,
    "direction": "en-hi"
  },
  "samanantar_1": {
    "total_rows": 1265714,
    "sampled": 1000,
    "direction": "en-hi"
  },
  "samanantar_2": {
    "total_rows": 1265713,
    "sampled": 1000,
    "direction": "en-hi"
  },
  "samanantar_3": {
    "total_rows": 1265713,
    "sampled": 1000,
    "direction": "en-hi"
  }
}


## Exact held-out hashes, sampled-train overlap only

In [10]:
paths=source_paths();held=[paths[k] for k in ('iit_validation','iit_test') if paths[k].exists()]
if held:
    forbidden=protected_hashes(held)
    train=paths['iit_train']
    if train.exists():
        sampled=reservoir_parquet(train,size=5000)['sample']
        overlap=sum(any(digest(t) in forbidden for t in extract_pair(row) if isinstance(t,str)) for row in sampled)
        print({'sampled_train_rows':len(sampled),'heldout_hashes':len(forbidden),'sampled_overlap_rows':overlap,'scope':'sampled training vs full held-out'})
else: print('SKIPPED: no official held-out splits')

{'sampled_train_rows': 5000, 'heldout_hashes': 6052, 'sampled_overlap_rows': 0, 'scope': 'sampled training vs full held-out'}


## Export factual manifest

In [12]:
manifest={'sources':report,'sample_statistics':samples,'note':'No accuracy, agreement, or leakage-free claim is made for missing datasets.'}
(ART/'dataset_manifest.json').write_text(json.dumps(manifest,indent=2));print('Saved manifest')

Saved manifest
